# Phase 5 fix — C-PD: retrain on cropped (leaf-only) images

**The damage:** full fine-tuning on FULL PlantDoc images taught the model to use the **background**. The healthy stages (PlantVillage leaf-attention, Paddy lesion-attention) were destroyed by the final PlantDoc stage. Three fixes failed because none forces the model to look at the leaf.

**The fix:** crop every training image to its leaf box → **no background left to learn from** → the model is forced to learn the leaf. This is the C-PD recipe (Singh et al. 2020, ~70% with honest attention).

Goal here is a model that **looks at the leaf** (fixes the damage). Accuracy ~70% is fine — a correct model at 70% beats a broken one at 72%.

- **Cell 3** proves the damage: Grad-CAM the current PlantDoc model on CLEAN PlantVillage images → it looks at background even there.
- **Cell 4** retrains on leaf crops (warm-started from the PlantVillage backbone).
- **Cell 5** tests: held-out crop accuracy + Grad-CAM (should now be on the leaf).

In [ ]:
# Cell 2 — clone repo + PlantDoc detection (boxes) repo + deps + HF login + GPU.
import os, shutil, subprocess, sys
REPO_PATH = "/content/iks-rag-thesis"
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
DET_PATH = "/content/PlantDoc-Object-Detection-Dataset"
DET_URL = "https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git"

os.chdir("/content")
shutil.rmtree(REPO_PATH, ignore_errors=True)
env = os.environ.copy(); env["GIT_LFS_SKIP_SMUDGE"] = "1"
r = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], env=env, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout); print(r.stderr); raise RuntimeError("clone iks failed")
if not os.path.isdir(DET_PATH):
    r = subprocess.run(["git", "clone", "--depth", "1", DET_URL, DET_PATH], env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr); raise RuntimeError("clone detection repo failed")
os.chdir(REPO_PATH); sys.path.insert(0, REPO_PATH)
print("repos ready")

DEPS = ["timm>=1.0", "albumentations>=1.4", "datasets>=2.20", "huggingface_hub>=0.24",
        "grad-cam>=1.5", "pydantic>=2.7", "opencv-python-headless", "matplotlib>=3.7"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], capture_output=True, text=True)
if r.returncode != 0:
    print("\n".join(r.stderr.splitlines()[-30:])); raise RuntimeError("pip failed")
print("deps installed")

from huggingface_hub import HfApi, login
login(); print("HF user:", HfApi().whoami().get("name"))
import torch
assert torch.cuda.is_available(), "Switch to T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 — PROVE THE DAMAGE: current PlantDoc model on CLEAN PlantVillage images.
# It should look at BACKGROUND even on clean images = features were distorted.
import random, matplotlib.pyplot as plt
from datasets import load_dataset
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from src.explain.gradcam import _preprocess_for_gradcam
from src.disease.infer import DiseaseInferenceEngine

old = DiseaseInferenceEngine(model_source="ankit-iiitdmj/iks-disease-plantdoc", device="cuda")
pv = load_dataset("ankit-iiitdmj/iks-plantvillage")
pv = pv["test" if "test" in pv else "train"]

def cam(eng, pil):
    t, rgb_u8, rgb_f = _preprocess_for_gradcam(pil, image_size=eng.image_size)
    mod = eng.model._module if hasattr(eng.model, "_module") else eng.model
    bb = eng.model.get_feature_extractor(); mod.eval()
    tt = t.to(next(mod.parameters()).device).requires_grad_(True)
    pred = eng.predict(pil).prediction
    g = GradCAM(model=mod, target_layers=[bb.blocks[-2]])(
        input_tensor=tt, targets=[ClassifierOutputTarget(int(pred.class_index))])[0]
    return show_cam_on_image(rgb_f, g, use_rgb=True), rgb_u8

random.seed(1)
for i in random.sample(range(len(pv)), 4):
    pil = pv[i]["image"].convert("RGB")
    ov, rgb = cam(old, pil)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(rgb); ax[0].set_title("clean PlantVillage leaf"); ax[0].axis("off")
    ax[1].imshow(ov); ax[1].set_title("PlantDoc model looks here"); ax[1].axis("off")
    plt.tight_layout(); plt.show()
print("If the heat is on background/edges even on these clean leaves -> damage confirmed.")

In [ ]:
# Cell 4 — RETRAIN on leaf crops (warm-start from PlantVillage backbone).
import json, os
from src.disease.train import CheckpointManager, train_one_stage, auto_batch_size
from src.disease.train_crop import (
    DEFAULT_CROP_REPO, build_crop_model, build_cropped_loaders,
)
from src.disease.config import DiseaseConfig
from src.utils.config import load_config
from pathlib import Path
import torch

# Find detection TRAIN folder.
cands = ["/content/PlantDoc-Object-Detection-Dataset/TRAIN", "/content/PlantDoc-Object-Detection-Dataset/train"]
TRAIN_DIR = next((c for c in cands if os.path.isdir(c)), None)
assert TRAIN_DIR, "detection TRAIN folder not found"

with open("data/splits/plantdoc/class_map.json") as f:
    class_map = json.load(f)

bs = auto_batch_size(380)
train_loader, val_loader, n_train = build_cropped_loaders(
    TRAIN_DIR, class_map, batch_size=bs, val_frac=0.1, num_workers=2,
)
print(f"cropped train samples: {n_train}, batch_size: {bs}")

model = build_crop_model(num_classes=27)
config = load_config(Path("configs/disease/default.yaml"), DiseaseConfig)

ckpt = CheckpointManager(DEFAULT_CROP_REPO)
ckpt.ensure_repo(private=True)

# Resume if a partial run exists.
prev = ckpt.try_load_latest()
start_epoch, history = 0, []
if prev is not None:
    start_epoch = int(prev.get("epoch", 0)); history = list(prev.get("history", []))
    model.load_state_dict(prev["model_state"], strict=False)
    print(f"resuming from epoch {start_epoch}")

result = train_one_stage(
    "finetune_plantdoc", train_loader, val_loader, model, config, ckpt,
    start_epoch=start_epoch, total_epochs=25, history=history, device="cuda",
)
print("\nBEST crop-retrain val acc:", round(result["best_val_acc"], 4))

In [ ]:
# Cell 5 — TEST: held-out crop accuracy + Grad-CAM (should now be on the leaf).
import os
from src.disease.train import evaluate
from src.disease.train_crop import build_cropped_test_loader, DEFAULT_CROP_REPO
from src.disease.infer import DiseaseInferenceEngine

# Held-out detection TEST crops (disjoint from TRAIN by the repo's split).
cands = ["/content/PlantDoc-Object-Detection-Dataset/TEST", "/content/PlantDoc-Object-Detection-Dataset/test"]
TEST_DIR = next((c for c in cands if os.path.isdir(c)), None)
test_loader, n_test = build_cropped_test_loader(TEST_DIR, class_map, batch_size=bs, num_workers=2)

# Load the crop-retrained model fresh from HF + evaluate on held-out crops.
new = DiseaseInferenceEngine(model_source=DEFAULT_CROP_REPO, device="cuda")
m = new.model._module if hasattr(new.model, "_module") else new.model
metrics = evaluate(m, test_loader, num_classes=27, device="cuda")
print("=" * 50)
print(f"Crop-retrained model — held-out crop test ({n_test} crops)")
print(f"  top-1 accuracy : {metrics.top1_accuracy:.1%}")
print(f"  macro F1       : {metrics.macro_f1():.3f}")
print("=" * 50)

# Grad-CAM: OLD (full-FT) vs NEW (crop-retrained) on the same PlantDoc images.
import matplotlib.pyplot as plt, random
pd_test = load_dataset("ankit-iiitdmj/iks-plantdoc", split="test")
random.seed(7)
for i in random.sample(range(len(pd_test)), 4):
    pil = pd_test[i]["image"].convert("RGB")
    ov_old, rgb = cam(old, pil)
    ov_new, _ = cam(new, pil)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
    ax[0].imshow(rgb); ax[0].set_title("input"); ax[0].axis("off")
    ax[1].imshow(ov_old); ax[1].set_title("OLD full-FT (background)"); ax[1].axis("off")
    ax[2].imshow(ov_new); ax[2].set_title("NEW crop-retrained (leaf?)"); ax[2].axis("off")
    plt.tight_layout(); plt.show()

## How to read it

| If we see... | Meaning |
|---|---|
| NEW heatmap on the **leaf** + acc ~70% | ✅ **Damage fixed** — model looks at the leaf. Ship this. |
| NEW heatmap still on background | crops weren't clean / detector boxes noisy — inspect a few crops |
| acc much < 70% | too few crops or class-map mismatch — check Cell 4 sample count |

The win here is **honest leaf attention**, not a higher number. ~70% with correct attention is the goal.